<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Moyenne par quart d'heure + jour de la semaine + mois  - RMSE public : 5.5146
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bilbiothèques
</div>

In [1]:
import pandas as pd

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des documents de données temporelles et de test
</div>

In [2]:
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Prédictions
</div>
On convertit les indexe en colonnes classiques

In [3]:
train = history_toulouse_train.reset_index()

On convertit les chaines de caractère en format date en on extrait le jour de la semaine, les heures et les minutes

In [4]:
train["mois"] = train["commit_at"].dt.month
train["jour"] = train["commit_at"].dt.dayofweek
train["heure"] = train["commit_at"].dt.hour
train["minutes"] = train["commit_at"].dt.minute

Moyenne par quart d'heure de chaque station

In [5]:
moyenne_station_mois = train.groupby(["station", "mois", "jour", "heure", "minutes"])["bikes"].mean().round().astype(int)

Moyennes de moins en moins précises en cas de valeurs manquantes

In [6]:
moyenne_station_quart_heure = train.groupby(["station", "jour", "heure", "minutes"])["bikes"].mean().round().astype(int)
moyenne_station_jour_heure = train.groupby(["station", "jour", "heure"])["bikes"].mean().round().astype(int)
moyenne_station_heure = train.groupby(["station", "heure"])["bikes"].mean().round().astype(int)
moyenne_station = train.groupby("station")["bikes"].mean().round().astype(int)

On extrait tous les indexes station et commit_at_index sous forme de liste

In [7]:
station_index = history_toulouse_test.index.get_level_values("station")
commit_at_index = history_toulouse_test.index.get_level_values("commit_at")

On associe chaque nom de colonne à une valeur numérique

In [8]:
decalage = {
    "bikes_+15min": 15,
    "bikes_+30min": 30,
    "bikes_+45min": 45,
    "bikes_+60min": 60,
    "bikes_+75min": 75,
    "bikes_+90min": 90,
    "bikes_+105min": 105,
    "bikes_+120min": 120,
}

On procède pour chaque décalage individuellement

In [9]:
for col, m in decalage.items():
    # Instant cible = commit_at + h minutes
    mois_cible = (commit_at_index + pd.Timedelta(minutes=m)).month
    jour_cible = (commit_at_index + pd.Timedelta(minutes=m)).dayofweek
    heure_cible = (commit_at_index + pd.Timedelta(minutes=m)).hour
    minute_cible = (commit_at_index + pd.Timedelta(minutes=m)).minute
    # On relie chaque station à son heure
    cle = list(zip(station_index, mois_cible, jour_cible, heure_cible, minute_cible))
    # On associe chaque (station_index, heure_cible, minute_cible) à la moyenne horaire du nombre de vélos
    valeurs = pd.Series(cle, index=history_toulouse_test.index).map(moyenne_station_mois)
    # En cas de valeur manquante, on utilise les moyennes calcuées avant
    cle = list(zip(station_index, jour_cible, heure_cible, minute_cible))
    valeurs = valeurs.fillna(pd.Series(cle, index=history_toulouse_test.index).map(moyenne_station_quart_heure))
    cle = list(zip(station_index, jour_cible, heure_cible))
    valeurs = valeurs.fillna(pd.Series(cle, index=history_toulouse_test.index).map(moyenne_station_jour_heure))
    cle = list(zip(station_index, heure_cible))
    valeurs = valeurs.fillna(pd.Series(cle, index=history_toulouse_test.index).map(moyenne_station_heure))
    valeurs = valeurs.fillna(pd.Series(station_index, index=history_toulouse_test.index).map(moyenne_station))
    # On sauvegarde le résultat en tant qu'entier
    history_toulouse_test[col] = valeurs.astype(int)

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Sauvegarde du résultat
</div>

In [10]:
history_toulouse_test.to_parquet("BA_prediction_5_quart_heure_semaine_mois.parquet", index=True)
history_toulouse_test

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,9,9,8,7,6,6,6,4
00002 - LAFAYETTE,2026-07-31 23:45:00,9,8,8,9,7,7,7,7
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,12,13,13,13,12,9,8,8
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,4,4,4,4,4,4,4,2
00003 - POMME,2025-01-05 05:45:00,3,3,3,3,3,2,2,3
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,0,0,0,0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0,0,0,0,0,0,0,0
